<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Matgl_mlearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install -q matgl

In [ ]:
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html

In [ ]:
# !pip uninstall dgl -y
# !pip install  dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html

In [ ]:
import os
if not os.path.exists('jarvis_leaderboard'):
  !git clone https://github.com/usnistgov/jarvis_leaderboard.git
os.chdir('jarvis_leaderboard')
!pip install -q -e .

In [ ]:
# !pip install numpy==1.23.5
# # then restart session

In [ ]:
!wget https://figshare.com/ndownloader/files/40357663 -O mlearn.json.zip

In [ ]:
!wget https://raw.githubusercontent.com/materialsvirtuallab/matgl/126ea69b17cc1b991b280ddbfb6358d9b8c1dd8a/requirements.txt -O requirements.txt
!pip install -r requirements.txt --ignore-installed --no-deps 2> install_errors.log


In [ ]:
!pip uninstall dgl -y
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.1/cu121/repo.html

In [ ]:
import json,zipfile
mlearn = json.loads(
        zipfile.ZipFile("mlearn.json.zip").read(
            "mlearn.json"
        )
    )

In [ ]:
%%time
import os
from jarvis.core.atoms import Atoms
import os
import shutil
import warnings
from ase.stress import voigt_6_to_full_3x3_stress
from functools import partial

import numpy as np
import pytorch_lightning as pl
from dgl.data.utils import split_dataset
#from mp_api.client import MPRester
from pytorch_lightning.loggers import CSVLogger
import numpy as np
import torch
from matgl.apps.pes import Potential
import zipfile,json, glob
import pandas as pd
import matgl
from matgl.ext.pymatgen import Structure2Graph, get_element_list
from matgl.graph.data import MGLDataset, MGLDataLoader, collate_fn_pes
from matgl.models import M3GNet
from matgl.utils.training import ModelLightningModule,PotentialLightningModule

# To suppress warnings for clearer output
warnings.simplefilter("ignore")

os.chdir('/content')
elements = ['Si']
device = "cpu"
max_epochs=100
models={}
for element in elements:

    benchmark_energies = (
        "jarvis_leaderboard/jarvis_leaderboard/benchmarks/AI/MLFF/mlearn_"
        + element
        + "_energy.json.zip"
    )
    temp_energies = benchmark_energies.split("/")[-1].split(".zip")[0]
    energies = json.loads(
        zipfile.ZipFile(benchmark_energies).read(temp_energies)
    )
    train_ids = list(energies["train"].keys())
    test_ids = list(energies["test"].keys())

    train_energies=[]
    train_forces=[]
    train_stresses=[]
    train_structures=[]
    for i in mlearn:
        if i["jid"] in train_ids:
            # print(i)
            train_energies.append(i["energy"])
            train_forces.append(i["forces"])
            train_stresses.append(voigt_6_to_full_3x3_stress(i['stresses']))
            atoms = Atoms.from_dict(i["atoms"])
            train_structures.append(atoms.pymatgen_converter())


    labels = {
        "energies": train_energies,
        "forces": train_forces,
        "stresses": train_stresses,
    }



    train_element_types = get_element_list(train_structures)
    converter = Structure2Graph(element_types=train_element_types, cutoff=5.0)
    train_data = MGLDataset(
        threebody_cutoff=4.0,
        structures=train_structures,
        converter=converter,
        labels=labels,
        include_line_graph=True,
        save_cache=False,
    )


    test_energies=[]
    test_forces=[]
    test_stresses=[]
    test_structures=[]
    tids = []
    for i in mlearn:
        if i["jid"] in test_ids:
            # print(i)
            test_energies.append(i["energy"])
            test_forces.append(i["forces"])
            test_stresses.append(voigt_6_to_full_3x3_stress(i['stresses']))
            atoms = Atoms.from_dict(i["atoms"])
            test_structures.append(atoms.pymatgen_converter())
            tids.append(i['jid'])

    labels = {
        "energies": test_energies,
        "forces": test_forces,
        "stresses": test_stresses,
    }



    element_types = get_element_list(test_structures)
    converter = Structure2Graph(element_types=element_types, cutoff=5.0)
    test_data = MGLDataset(
        threebody_cutoff=4.0,
        structures=test_structures,
        converter=converter,
        labels=labels,
        include_line_graph=True,
        save_cache=False,
    )






    my_collate_fn = partial(collate_fn_pes, include_line_graph=True)
    train_loader, val_loader, test_loader = MGLDataLoader(
        train_data=train_data,
        val_data=test_data,
        test_data=test_data,
        collate_fn=my_collate_fn,
        batch_size=2,
        num_workers=0,
        generator=torch.Generator(device=device),
    )




In [ ]:
%%time
# from https://github.com/materialsvirtuallab/matgl/blob/main/tests/utils/test_training.py#L89
import numpy as np
from functools import partial
from ase.stress import voigt_6_to_full_3x3_stress
import lightning as pl
import numpy as np
import pytest
import torch.backends.mps
from dgl.data.utils import split_dataset
from matgl.ext.pymatgen import Structure2Graph, get_element_list
from matgl.graph.data import MGLDataLoader, MGLDataset, collate_fn_graph, collate_fn_pes
from matgl.models import CHGNet, M3GNet, MEGNet, SO3Net, TensorNet
from matgl.utils.training import ModelLightningModule, PotentialLightningModule, xavier_init
from pymatgen.core import Lattice, Structure
# def test_m3gnet_training(structures=[],energies=[],forces=[],stresses=[],element_types=[]):
device = "mps" if torch.backends.mps.is_available() else "cpu"

model = M3GNet(element_types=element_types, is_intensive=False)
lit_model = PotentialLightningModule(model=model, stress_weight=0.0001, include_line_graph=True)
model_save_path = "trained_model"
# We will use CPU if MPS is available since there is a serious bug.
trainer = pl.Trainer(max_epochs=20, accelerator=device, inference_mode=False)

trainer.fit(model=lit_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
trainer.test(lit_model, dataloaders=test_loader)

lit_model.model.save(model_save_path)





In [ ]:
from matgl.ext.ase import PESCalculator

trained_model = matgl.load_model(path="trained_model")
matgl_calc = PESCalculator(trained_model)

In [ ]:
!wget https://figshare.com/ndownloader/files/40357663 -O mlearn.json.zip

In [ ]:
import numpy as np
import torch
from matgl.apps.pes import Potential
import zipfile,json, glob
import pandas as pd
def get_matgl_pred(atoms=None,model=None):
    atoms=atoms.ase_converter()
    atoms.set_calculator(matgl_calc)
    energy = atoms.get_potential_energy()
    forces = atoms.get_forces()
    stress = atoms.get_stress()
    return energy, forces, stress



df = pd.DataFrame(
    json.loads(
        zipfile.ZipFile("mlearn.json.zip").read(
            "mlearn.json"
        )
    )
)
print(df)
i_model = trainer.model.model.model
for i in glob.glob("/content/jarvis_leaderboard/jarvis_leaderboard/benchmarks/AI/MLFF/*energy*.zip"):
    if "mlearn" in i and element in i:
        fname_e = (
            "AI-MLFF-energy-"
            + i.split("/")[-1].split("_energy.json.zip")[0]
            + "-test-mae.csv"
        )
        fname_f = (
            "AI-MLFF-forces-"
            + i.split("/")[-1].split("_energy.json.zip")[0]
            + "-test-multimae.csv"
        )
        fname_s = (
            "AI-MLFF-stresses-"
            + i.split("/")[-1].split("_energy.json.zip")[0]
            + "-test-multimae.csv"
        )
        f_e = open(fname_e, "w")
        f_f = open(fname_f, "w")
        f_s = open(fname_s, "w")

        f_e.write("id,target,prediction\n")
        f_f.write("id,target,prediction\n")
        f_s.write("id,target,prediction\n")

        print(i)
        dat = json.loads(
            zipfile.ZipFile(i).read(i.split("/")[-1].split(".zip")[0])
        )
        print(dat["test"])
        for key, val in dat["test"].items():
            entry = df[df["jid"] == key]
            atoms = Atoms.from_dict(entry.atoms.values[0])
            # print(key,val,df[df['jid']==key],atoms)
            # energy,forces=get_alignn_forces(atoms)
            energy, forces, stress = get_matgl_pred(atoms=atoms,model=trained_model)
            print(key, val, energy, atoms.num_atoms)
            line = key + "," + str(entry.energy.values[0])+","+str(energy) + "\n"
            f_e.write(line)
            line = (
                key
                + ","
                + str(";".join(map(str, np.array(entry.forces.values[0]).flatten())))
                +","
                + str(";".join(map(str, np.array(forces).flatten())))
                + "\n"
            )
            f_f.write(line)
            line = (
                key
                + ","
                + str(";".join(map(str, np.array(entry.stresses.values[0]).flatten())))
                +","
                + str(";".join(map(str, np.array(stress).flatten())))
                + "\n"
            )
            f_s.write(line)
        f_e.close()
        f_f.close()
        f_s.close()
        zname = fname_e + ".zip"
        with zipfile.ZipFile(zname, "w") as myzip:
            myzip.write(fname_e)

        zname = fname_f + ".zip"
        with zipfile.ZipFile(zname, "w") as myzip:
            myzip.write(fname_f)

        zname = fname_s + ".zip"
        with zipfile.ZipFile(zname, "w") as myzip:
            myzip.write(fname_s)




In [ ]:
!ls -altr

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
%matplotlib inline
df = pd.read_csv('AI-MLFF-energy-mlearn_Si-test-mae.csv')
plt.plot(df.target,df.prediction,'.')
print(mean_absolute_error(df.target,df.prediction))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
df = pd.read_csv('AI-MLFF-energy-mlearn_Si-test-mae.csv')
#plt.plot(df.target,df.prediction,'.')
print('Energy',mean_absolute_error(df.target,df.prediction))
df = pd.read_csv('AI-MLFF-forces-mlearn_Si-test-multimae.csv.zip')
target = np.concatenate(df.target.apply(lambda x:np.array(x.split(';'),dtype='float')).values)
pred= np.concatenate(df.prediction.apply(lambda x:np.array(x.split(';'),dtype='float')).values)
print('Forces',mean_absolute_error(target,pred))
plt.plot(target,pred,'.')


In [ ]:
# df = pd.read_csv('AI-MLFF-forces-mlearn_Si-test-multimae.csv.zip')
# plt.plot(np.array(df.target.values[0].split(';'),dtype='float'),np.array(df.prediction.values[0].split(';'),dtype='float'),'.')
# print(mean_absolute_error(np.array(df.target.values[0].split(';'),dtype='float'),np.array(df.prediction.values[0].split(';'),dtype='float')))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('AI-MLFF-stresses-mlearn_Si-test-multimae.csv.zip')
target = np.concatenate(df.target.apply(lambda x:np.array(x.split(';'),dtype='float')).values)
pred= np.concatenate(df.prediction.apply(lambda x:np.array(x.split(';'),dtype='float')).values)
print('Stress',mean_absolute_error(target,pred))
plt.plot(target,pred,'.')


In [ ]:
# df = pd.read_csv('AI-MLFF-stresses-mlearn_Si-test-multimae.csv.zip')
# plt.plot(np.array(df.target.values[0].split(';'),dtype='float'),np.array(df.prediction.values[0].split(';'),dtype='float'),'.')
# print(mean_absolute_error(np.array(df.target.values[0].split(';'),dtype='float'),np.array(df.prediction.values[0].split(';'),dtype='float')))

In [ ]:
!pip install -q jarvis-tools

In [ ]:
from ase.eos import EquationOfState
from ase.units import kJ
from jarvis.db.figshare import get_jid_data
from jarvis.core.atoms import Atoms
import matplotlib.pyplot as plt
%matplotlib inline
calc= matgl_calc
atoms=Atoms.from_dict(get_jid_data(jid='JVASP-1002',dataset='dft_3d')['atoms']) #.get_conventional_atoms
#dx=np.arange(-0.1, 0.1, 0.01)
dx=np.arange(-0.05, 0.05, 0.01)
y = []
vol = []
for i in dx:
    s1 = atoms.strain_atoms(i)
    ase_atoms = s1.ase_converter()
    ase_atoms.calc = calc
    energy = ase_atoms.get_potential_energy()
    y.append(energy)
    vol.append(s1.volume)
x = np.array(dx)
y = np.array(y)
eos = EquationOfState(vol, y, eos="murnaghan")
v0, e0, B = eos.fit()
kv = B / kJ * 1.0e24  # , 'GPa')
print(kv)
plt.plot(vol,y)
plt.xlabel('Volume ($\AA^3$)')
plt.ylabel('Total energy (eV)')


In [ ]:
#Check translationally equivariant
from ase import Atom, Atoms
import numpy as np
import matplotlib.pyplot as plt


A = 2.7223*np.array([[1,1,0],[1,0,1],[0,1,1]])
atoms_si = Atoms([Atom('Si', (0, 0, 0)), Atom('Si', (0.25, 0.25, 0.25))], cell = A, pbc=True)

atoms_si.set_tags(np.ones(len(atoms_si)))
atoms_si.calc = calc

atoms_si.set_scaled_positions(np.array([[0,0,0],[0.25, 0.25, 0.25]]))

#"""
en = []
dist = []
for a in np.linspace(-0.1,  0.1, 51):
    atoms_si.set_scaled_positions(np.array([[a,0,0],[0.25, 0.25, 0.25]]))
    e = atoms_si.get_potential_energy()
    en.append(e)
    dist.append(np.sign(a)*np.sqrt(np.sum(np.dot(A, [a,0,0])**2)))
en_111_more = []
dist_111_more = []
for a in np.linspace(-0.1,  0.1, 201):
    atoms_si.set_scaled_positions(np.array([[a,a,a],[0.25, 0.25, 0.25]]))
    e = atoms_si.get_potential_energy()
    en_111_more.append(e)
    dist_111_more.append(np.sign(a)*np.sqrt(np.sum(np.dot(A, [a,a,a])**2)))
plt.plot(dist,en)
plt.plot(dist_111_more,en_111_more)
#plt.ylim([-10.7,-10])
# plt.savefig('kevin.png')
# plt.close()
#"""




In [ ]:
#Check rotationally equivariant
atoms_si = Atoms([Atom('Si', (0, 0, 0)), Atom('Si', (0.25, 0.25, 0.25))], cell = A, pbc=True)
atoms_si.calc = calc
#translations
en_trans = []
for a in np.linspace(-0.1,  0.1, 51):
    atoms_si.set_scaled_positions(np.array([[a,a,a],[a+0.25, a+0.25, a+0.25]]))
    e = atoms_si.get_potential_energy()
    en_trans.append(e)
plt.plot(en_trans)

In [ ]:
%%time
import ase,os
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.nvtberendsen import NVTBerendsen
from jarvis.db.figshare import get_jid_data
from jarvis.core.atoms import Atoms, ase_to_atoms
def ensure_cell_size(ase_atoms, min_size):
    """Ensure that all cell dimensions are at least min_size."""
    cell_lengths = ase_atoms.get_cell().lengths()
    scale_factors = np.ceil(min_size / cell_lengths).astype(int)
    supercell_dims = [max(1, scale) for scale in scale_factors]
    return supercell_dims

calculator = calc
jid = 'JVASP-1002'
atoms=Atoms.from_dict(get_jid_data(jid=jid,dataset='dft_3d')['atoms']).get_conventional_atoms
relaxed_atoms=atoms
ase_atoms = atoms.ase_converter()
dim = ensure_cell_size(ase_atoms, min_size=10.0)
supercell = relaxed_atoms.make_supercell_matrix(dim)
ase_atoms = supercell.ase_converter()
ase_atoms.calc = calculator

dt = 1 * ase.units.fs
temp0, nsteps0 = 3500, 1000
temp1, nsteps1 = 300, 2000
taut = 20 * ase.units.fs
if not os.path.exists("output_melt"):
    os.mkdir("output_melt")
trj = os.path.join("output_melt", f"{jid}_melt.traj")

# Initialize velocities and run the first part of the MD simulation


MaxwellBoltzmannDistribution(ase_atoms, temp0 * ase.units.kB)
dyn = NVTBerendsen(ase_atoms, dt, temp0, taut=taut, trajectory=trj)

def myprint():
    message = f"time={dyn.get_time() / ase.units.fs: 5.0f} fs T={ase_atoms.get_temperature(): 3.0f} K"
    print(message)
dyn.attach(myprint, interval=20)
dyn.run(nsteps0)

# Cool down to room temperature
dyn.set_temperature(temp1)
dyn.run(nsteps1)

# Convert back to JARVIS atoms and save the final structure
final_atoms = ase_to_atoms(ase_atoms)
poscar_filename = os.path.join(
    "output_melt", f"POSCAR_{jid}_quenched_{'matgl'}.vasp"
)
from ase.io import write

write(poscar_filename, final_atoms.ase_converter(), format="vasp")
# self.log(f"MD simulation completed. Final structure saved to {poscar_filename}")
# self.job_info["quenched_atoms"] = final_atoms.to_dict()


In [ ]:
!ls output_melt

In [ ]:
# RDF
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from jarvis.core.atoms import Atoms
atoms=Atoms.from_poscar('output_melt/POSCAR_JVASP-1002_quenched_matgl.vasp')

dists=[]
for i in atoms.get_all_neighbors(r=10):
    for j in i:
      dists.append(j[2])
dists=np.array(dists)
hist, bins = np.histogram(dists, bins=np.arange(0.1, 10.2, 0.1))
number_density = atoms.num_atoms / atoms.volume
shell_vol = (
            4.0
            / 3.0
            * np.pi
            * (np.power(bins[1:], 3) - np.power(bins[:-1], 3))
        )

rdf = (
            hist / shell_vol / number_density / atoms.num_atoms
        )
plt.plot(bins[:-1], rdf)
plt.xlim([0,10])
plt.xlabel('Distance (Angstrom)')
plt.ylabel('RDF')
plt.show()